# Marketing A/B Test: Ad Campaign Effectiveness

This notebook compares conversion rates for users assigned to the ad and public service announcement (PSA) groups. It also explores how observed ad exposure relates to conversion.

**Primary question:** Is the conversion-rate difference between the provided test groups statistically distinguishable from zero?

The exposure and weekday breakdowns are descriptive. Exposure occurs after group assignment, so those comparisons do not estimate the causal effect of showing a specific number of ads.


In [1]:
from pathlib import Path
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Find the project root whether Jupyter starts in the repo root or notebooks/.
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "marketing_AB.csv").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Download marketing_AB.csv into data/; see data/README.md for the source."
    )

df = pd.read_csv(PROJECT_ROOT / "data" / "marketing_AB.csv")
# This column is an exported row index, not a measured feature.
df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed:")])
print("Columns:", list(df.columns))


Columns: ['user id', 'test group', 'converted', 'total ads', 'most ads day', 'most ads hour']


## 1. Data checks

Confirm the row count, missing values, duplicate users, group labels, and outcome values before comparing groups.


In [2]:
print(f"Rows: {len(df):,}")
print(f"Unique users: {df['user id'].nunique():,}")
print(f"Duplicate user IDs: {df['user id'].duplicated().sum():,}")
print(f"Missing cells: {df.isna().sum().sum():,}")
print("Groups:", sorted(df["test group"].unique()))
print("Outcome values:", sorted(df["converted"].unique()))

assert df["user id"].is_unique, "Expected one row per user."
assert df.isna().sum().sum() == 0, "Resolve missing data before analysis."
assert set(df["test group"].unique()) == {"ad", "psa"}, "Unexpected test-group labels."
print("Observed allocation:")
print((df["test group"].value_counts(normalize=True).mul(100).round(2).astype(str) + "%").to_string())


Rows: 588,101
Unique users: 588,101
Duplicate user IDs: 0
Missing cells: 0
Groups: ['ad', 'psa']
Outcome values: [np.False_, np.True_]
Observed allocation:
test group
ad     96.0%
psa     4.0%


The dataset uses a strongly unequal allocation (about 96% ads and 4% PSA). Check the original experiment plan before treating that ratio as expected or diagnosing sample-ratio mismatch.


## 2. Primary metric: conversion rate

Conversion rate is the number of converted users divided by users in each group. The estimated absolute effect is Ads minus PSA; relative lift is that difference divided by the PSA conversion rate.


In [3]:
group_summary = (
    df.groupby("test group")["converted"]
    .agg(users="size", conversions="sum", conversion_rate="mean")
    .reindex(["ad", "psa"])
)
group_summary["allocation_share"] = group_summary["users"] / len(df)
print(group_summary.to_string(formatters={
    "conversion_rate": "{:.4%}".format,
    "allocation_share": "{:.2%}".format,
}))


             users  conversions conversion_rate allocation_share
test group                                                      
ad          564577        14423         2.5547%           96.00%
psa          23524          420         1.7854%            4.00%


## 3. Two-proportion z-test and confidence interval

The two-sided z-test uses the pooled standard error under the null hypothesis of equal conversion rates. The interval below is a 95% Newcombe interval based on Wilson intervals for the two independent proportions.


In [4]:
ad = group_summary.loc["ad"]
psa = group_summary.loc["psa"]
p_ad, p_psa = ad["conversion_rate"], psa["conversion_rate"]
n_ad, n_psa = int(ad["users"]), int(psa["users"])
x_ad, x_psa = int(ad["conversions"]), int(psa["conversions"])
difference = p_ad - p_psa
relative_lift = difference / p_psa

pooled_rate = (x_ad + x_psa) / (n_ad + n_psa)
pooled_se = math.sqrt(pooled_rate * (1 - pooled_rate) * (1 / n_ad + 1 / n_psa))
z_stat = difference / pooled_se
p_value = math.erfc(abs(z_stat) / math.sqrt(2))

def wilson_interval(successes, total, z=1.96):
    rate = successes / total
    denom = 1 + z**2 / total
    center = (rate + z**2 / (2 * total)) / denom
    margin = z * math.sqrt(rate * (1 - rate) / total + z**2 / (4 * total**2)) / denom
    return center - margin, center + margin

ad_low, ad_high = wilson_interval(x_ad, n_ad)
psa_low, psa_high = wilson_interval(x_psa, n_psa)
ci_low = difference - math.sqrt((p_ad - ad_low)**2 + (psa_high - p_psa)**2)
ci_high = difference + math.sqrt((ad_high - p_ad)**2 + (p_psa - psa_low)**2)

print(f"Ads conversion rate: {p_ad:.4%} ({x_ad:,}/{n_ad:,})")
print(f"PSA conversion rate: {p_psa:.4%} ({x_psa:,}/{n_psa:,})")
print(f"Absolute difference: {difference:.4%} ({difference * 100:.3f} percentage points)")
print(f"Relative lift vs PSA: {relative_lift:.2%}")
print(f"z statistic: {z_stat:.3f}")
print(f"Two-sided p-value: {p_value:.3e}")
print(f"95% Newcombe CI for difference: [{ci_low:.4%}, {ci_high:.4%}]")


Ads conversion rate: 2.5547% (14,423/564,577)
PSA conversion rate: 1.7854% (420/23,524)
Absolute difference: 0.7692% (0.769 percentage points)
Relative lift vs PSA: 43.09%
z statistic: 7.370
Two-sided p-value: 1.705e-13
95% Newcombe CI for difference: [0.5873%, 0.9360%]


### Interpretation

The observed ad-group conversion rate is higher than the PSA-group rate, and the two-sided test gives strong evidence against equal rates. This supports a campaign effect only if the groups were assigned as intended, outcomes were measured consistently, and the user-level observations are independent. Statistical significance does not establish profitability; campaign costs and revenue are not included.


## 4. Exploratory breakdowns

The following tables show conversion by the day with the most ads and by ad-exposure bands. These variables describe observed exposure, not randomized treatments. Use them to generate questions, not to claim that a particular day or frequency caused more conversions.


In [5]:
weekday_summary = (
    df.groupby(["test group", "most ads day"])["converted"]
    .agg(users="size", conversions="sum", conversion_rate="mean")
    .reset_index()
)
print(weekday_summary.to_string(index=False, formatters={"conversion_rate": "{:.3%}".format}))

exposure_bins = [0, 5, 15, 30, 50, 100, np.inf]
exposure_labels = ["1-5", "6-15", "16-30", "31-50", "51-100", "101+"]
ad_users = df.loc[df["test group"] == "ad"].copy()
ad_users["exposure_band"] = pd.cut(
    ad_users["total ads"], bins=exposure_bins, labels=exposure_labels, include_lowest=True
)
exposure_summary = (
    ad_users.groupby("exposure_band", observed=False)["converted"]
    .agg(users="size", conversions="sum", conversion_rate="mean")
)
print("\nAd group only — conversion by observed ad-exposure band:")
print(exposure_summary.to_string(formatters={"conversion_rate": "{:.3%}".format}))


test group most ads day  users  conversions conversion_rate
        ad       Friday  88805         1995          2.246%
        ad       Monday  83571         2778          3.324%
        ad     Saturday  78802         1679          2.131%
        ad       Sunday  82332         2027          2.462%
        ad     Thursday  79077         1711          2.164%
        ad      Tuesday  74572         2270          3.044%
        ad    Wednesday  77418         1963          2.536%
       psa       Friday   3803           62          1.630%
       psa       Monday   3502           79          2.256%
       psa     Saturday   2858           40          1.400%
       psa       Sunday   3059           63          2.059%
       psa     Thursday   3905           79          2.023%
       psa      Tuesday   2907           42          1.445%
       psa    Wednesday   3490           55          1.576%

Ad group only — conversion by observed ad-exposure band:
                users  conversions convers

## 5. Visual summary

The exposure panel is descriptive and restricted to the ad group. Higher observed exposure may reflect user engagement or time in the campaign; this chart does not identify the causal effect of increasing ad frequency.


In [6]:
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# A: Primary test result
plot_groups = group_summary.reset_index()
wilson_bounds = [wilson_interval(int(row["conversions"]), int(row["users"])) for _, row in plot_groups.iterrows()]
rate_errors = np.array([[r - lo, hi - r] for r, (lo, hi) in zip(plot_groups["conversion_rate"], wilson_bounds)]).T
axes[0].bar(["Ads", "PSA"], plot_groups["conversion_rate"], yerr=rate_errors, capsize=5, color=["#2563eb", "#94a3b8"])
axes[0].set_title("Conversion rate by group")
axes[0].set_ylabel("Conversion rate")
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.1%}"))
for i, row in plot_groups.iterrows():
    axes[0].text(i, row["conversion_rate"], f"{row['conversion_rate']:.2%}", ha="center", va="bottom")

# B: Descriptive association by ad frequency among treatment users
exp_plot = exposure_summary.reset_index()
axes[1].plot(exp_plot["exposure_band"].astype(str), exp_plot["conversion_rate"], marker="o", color="#0f766e")
axes[1].set_title("Exposure association (descriptive)")
axes[1].set_xlabel("Observed ads per user")
axes[1].set_ylabel("Conversion rate")
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
axes[1].tick_params(axis="x", rotation=35)

# C: Descriptive weekday differences, shown separately by group
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
for label, color in [("ad", "#2563eb"), ("psa", "#94a3b8")]:
    part = weekday_summary.loc[weekday_summary["test group"] == label].set_index("most ads day").reindex(weekday_order)
    axes[2].plot(weekday_order, part["conversion_rate"], marker="o", label=label.upper(), color=color)
axes[2].set_title("Conversion by highest-exposure day")
axes[2].set_ylabel("Conversion rate")
axes[2].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.1%}"))
axes[2].tick_params(axis="x", rotation=45)
axes[2].legend(frameon=False)

fig.suptitle("Marketing A/B test: campaign result and exploratory segments", y=1.03, fontsize=14)
fig.tight_layout()
figure_path = PROJECT_ROOT / "outputs" / "figures" / "marketing_ab_analysis.png"
figure_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(figure_path, dpi=180, bbox_inches="tight")
plt.show()


## Conclusion

The ad group has a higher observed conversion rate than the PSA group. The experiment’s allocation plan, independence assumptions, and business economics should be checked before making a rollout or budget decision. The frequency analysis is observational; this dataset does not establish an optimal frequency cap or ad-fatigue threshold.
